# Hash Tables and Bloom Filters

This notebook covers hash-based data structures that provide **O(1) average-case** operations for insert, search, and delete.

## Table of Contents
1. [Hash Functions](#1-hash-functions)
2. [Hash Tables - Direct Addressing](#2-hash-tables---direct-addressing)
3. [Collision Resolution](#3-collision-resolution)
4. [Load Factor & Rehashing](#4-load-factor--rehashing)
5. [Bloom Filters](#5-bloom-filters)

---

[← Previous: Red-Black Trees](../../Tier_4_Algorithms_and_Data_Structures/05_Tree_Structures/03_red_black_trees.ipynb) | [Next: Naive Pattern Matching Algorithm →](../../Tier_4_Algorithms_and_Data_Structures/07_String_Algorithms/01_naive_pattern_matching.ipynb)

---

---

## 1. Hash Functions

### What is a Hash Function?

A **hash function** maps data of arbitrary size to fixed-size values (hash codes). It converts keys into array indices for fast lookups.

```
key ──────► h(key) ──────► index
"hello"      hash()        42
"world"      hash()        17
12345        hash()        85
```

### Properties of a Good Hash Function

| Property | Description |
|----------|-------------|
| **Deterministic** | Same input always produces same output |
| **Uniform Distribution** | Spreads keys evenly across the table |
| **Fast Computation** | O(1) time to compute hash value |
| **Minimizes Collisions** | Different keys rarely map to same index |

### Why O(1) Average Lookup?

With a good hash function and proper table size:
1. Computing `h(key)` takes O(1) time
2. Direct array access at index `h(key)` takes O(1) time
3. If load factor is kept low, collisions are rare → O(1) on average

**Worst case O(n)** occurs when all keys hash to the same index (pathological case).

In [ ]:
# Simple Hash Function Examples

def simple_mod_hash(key: int, table_size: int) -> int:
    """
    Basic modulo hash function for integers.
    
    Args:
        key: Integer key to hash
        table_size: Size of the hash table (preferably prime)
    
    Returns:
        Hash index in range [0, table_size-1]
    """
    return key % table_size


def string_hash(key: str, table_size: int) -> int:
    """
    Polynomial rolling hash for strings.
    
    Uses base 31 (common choice for lowercase strings).
    h(s) = s[0] + s[1]*31 + s[2]*31^2 + ... + s[n]*31^n
    
    Args:
        key: String key to hash
        table_size: Size of the hash table
    
    Returns:
        Hash index in range [0, table_size-1]
    """
    hash_value = 0
    base = 31
    for char in key:
        hash_value = hash_value * base + ord(char)
    return hash_value % table_size


def multiplication_hash(key: int, table_size: int) -> int:
    """
    Multiplication method hash (Knuth's suggestion).
    
    h(k) = floor(m * (k * A mod 1))
    where A = (√5 - 1) / 2 ≈ 0.6180339887 (golden ratio conjugate)
    
    Args:
        key: Integer key to hash
        table_size: Size of the hash table (m)
    
    Returns:
        Hash index in range [0, table_size-1]
    """
    A = 0.6180339887  # (sqrt(5) - 1) / 2
    return int(table_size * ((key * A) % 1))


# Demonstrate hash functions
print("=== Hash Function Examples ===")
print(f"\nTable size: 7 (prime)")
print(f"\nModulo hash:")
for key in [15, 22, 8, 1, 29, 36]:
    print(f"  h({key}) = {key} % 7 = {simple_mod_hash(key, 7)}")

print(f"\nString hash (table size 10):")
for word in ["hello", "world", "hash", "table"]:
    print(f"  h('{word}') = {string_hash(word, 10)}")

---

## 2. Hash Tables - Direct Addressing

### Concept

A hash table uses an array where elements are stored at indices determined by a hash function.

```
Direct Addressing (no collisions):

Keys: {15, 3, 22, 8}
h(key) = key % 10

Index:  0    1    2    3    4    5    6    7    8    9
      ┌────┬────┬────┬────┬────┬────┬────┬────┬────┬────┐
      │    │    │ 22 │  3 │    │ 15 │    │    │  8 │    │
      └────┴────┴────┴────┴────┴────┴────┴────┴────┴────┘
              ↑    ↑         ↑              ↑
         22%10=2  3%10=3   15%10=5       8%10=8
```

### Complexity Analysis

| Operation | Average Case | Worst Case | Notes |
|-----------|--------------|------------|-------|
| Insert | O(1) | O(n) | Worst when all keys collide |
| Search | O(1) | O(n) | Worst when searching collision chain |
| Delete | O(1) | O(n) | Must handle collision structure |
| Space | O(n) | O(n) | n = number of stored elements |

In [ ]:
class SimpleHashTable:
    """
    Simple hash table without collision handling.
    
    Demonstrates basic hash table concept - will overwrite on collision.
    NOT suitable for production use.
    """
    
    def __init__(self, size: int = 10):
        """
        Initialize hash table with given size.
        
        Args:
            size: Number of slots in the table
        """
        self.size = size
        self.keys = [None] * size
        self.values = [None] * size
    
    def _hash(self, key) -> int:
        """Compute hash index for a key."""
        return hash(key) % self.size
    
    def put(self, key, value):
        """Insert or update a key-value pair."""
        index = self._hash(key)
        self.keys[index] = key
        self.values[index] = value
    
    def get(self, key):
        """Retrieve value by key. Returns None if not found."""
        index = self._hash(key)
        if self.keys[index] == key:
            return self.values[index]
        return None
    
    def display(self):
        """Print the hash table contents."""
        print("Index | Key    | Value")
        print("-" * 25)
        for i in range(self.size):
            key_str = str(self.keys[i]) if self.keys[i] is not None else "empty"
            val_str = str(self.values[i]) if self.values[i] is not None else "-"
            print(f"  {i}   | {key_str:6} | {val_str}")


# Demonstrate simple hash table
print("=== Simple Hash Table Demo ===")
ht = SimpleHashTable(7)

# Insert some values
ht.put(15, "apple")
ht.put(3, "banana")
ht.put(8, "cherry")

print("\nAfter inserting 15, 3, 8:")
ht.display()

print(f"\nget(15) = {ht.get(15)}")
print(f"get(3) = {ht.get(3)}")
print(f"get(99) = {ht.get(99)}")

---

## 3. Collision Resolution

A **collision** occurs when two different keys hash to the same index. There are two main strategies to handle collisions:

### 3.1 Chaining (Closed Addressing)

Each slot contains a linked list of all elements that hash to that index.

```
Hash function: h(key) = key % 7

Insert: 15, 22, 8, 1, 29, 36

Calculations:
  15 % 7 = 1    22 % 7 = 1    8 % 7 = 1
   1 % 7 = 1    29 % 7 = 1   36 % 7 = 1

All hash to index 1! (pathological case for demonstration)

Index:  0    1    2    3    4    5    6
      ┌────┬────┬────┬────┬────┬────┬────┐
      │    │ ●  │    │    │    │    │    │
      └────┴─│──┴────┴────┴────┴────┴────┘
             │
             ↓
           [15] → [22] → [8] → [1] → [29] → [36] → null

Insert/Search: Walk the chain at index h(key)
```

### 3.2 Open Addressing

All elements are stored in the array itself. On collision, probe for next available slot.

#### Linear Probing

```
h(key, i) = (h(key) + i) % m

Insert 15, 22, 29 (all hash to 1)

Step 1: h(15)=1, slot 1 empty → insert at 1
Index:  0    1    2    3    4    5    6
      ┌────┬────┬────┬────┬────┬────┬────┐
      │    │ 15 │    │    │    │    │    │
      └────┴────┴────┴────┴────┴────┴────┘

Step 2: h(22)=1, collision! probe i=1 → slot 2 empty → insert at 2
Index:  0    1    2    3    4    5    6
      ┌────┬────┬────┬────┬────┬────┬────┐
      │    │ 15 │ 22 │    │    │    │    │
      └────┴────┴────┴────┴────┴────┴────┘
            ↑────↑
            collision, move right

Step 3: h(29)=1 → try 2 (full) → try 3 → insert at 3
Index:  0    1    2    3    4    5    6
      ┌────┬────┬────┬────┬────┬────┬────┐
      │    │ 15 │ 22 │ 29 │    │    │    │
      └────┴────┴────┴────┴────┴────┴────┘
            ↑────↑────↑
            probe sequence
```

#### Quadratic Probing

```
h(key, i) = (h(key) + c₁*i + c₂*i²) % m

Typically: h(key, i) = (h(key) + i²) % m

Probes: +0, +1, +4, +9, +16, ...
Reduces primary clustering but may not visit all slots
```

#### Double Hashing

```
h(key, i) = (h₁(key) + i * h₂(key)) % m

h₁(key) = key % m
h₂(key) = 1 + (key % (m-1))  # must never return 0

Best distribution but requires two hash functions
```

### Comparison: Chaining vs Open Addressing

| Aspect | Chaining | Open Addressing |
|--------|----------|----------------|
| **Memory** | Extra for pointers | No extra (in-place) |
| **Cache** | Poor (pointer chasing) | Better (contiguous) |
| **Load factor** | Can exceed 1.0 | Must stay < 1.0 |
| **Delete** | Easy | Needs tombstones |
| **Clustering** | No clustering | Primary/secondary clustering |

In [ ]:
# Linked List Node for Chaining
class Node:
    """
    Node for linked list used in chaining collision resolution.
    
    Attributes:
        key: The key stored in this node
        value: The value associated with the key
        next: Reference to the next node in the chain
    """
    
    def __init__(self, key, value):
        self.key = key
        self.value = value
        self.next = None


class HashTableChaining:
    """
    Hash table with separate chaining collision resolution.
    
    Each bucket contains a linked list of key-value pairs that
    hash to the same index.
    
    Attributes:
        size: Number of buckets in the table
        buckets: Array of linked list heads (one per bucket)
        count: Number of key-value pairs stored
    """
    
    def __init__(self, size: int = 7):
        """
        Initialize hash table with given number of buckets.
        
        Args:
            size: Number of buckets (preferably prime for better distribution)
        """
        self.size = size
        self.buckets = [None] * size
        self.count = 0
    
    def _hash(self, key) -> int:
        """Compute bucket index for a key."""
        return hash(key) % self.size
    
    def put(self, key, value) -> None:
        """
        Insert or update a key-value pair.
        
        Args:
            key: Key to insert
            value: Value associated with the key
        """
        index = self._hash(key)
        
        # Search for existing key in chain
        current = self.buckets[index]
        while current:
            if current.key == key:
                current.value = value  # Update existing
                return
            current = current.next
        
        # Key not found, insert at head of chain
        new_node = Node(key, value)
        new_node.next = self.buckets[index]
        self.buckets[index] = new_node
        self.count += 1
    
    def get(self, key):
        """
        Retrieve value by key.
        
        Args:
            key: Key to search for
        
        Returns:
            Value associated with key, or None if not found
        """
        index = self._hash(key)
        current = self.buckets[index]
        
        while current:
            if current.key == key:
                return current.value
            current = current.next
        
        return None
    
    def delete(self, key) -> bool:
        """
        Remove a key-value pair from the table.
        
        Args:
            key: Key to remove
        
        Returns:
            True if key was found and removed, False otherwise
        """
        index = self._hash(key)
        current = self.buckets[index]
        prev = None
        
        while current:
            if current.key == key:
                if prev:
                    prev.next = current.next
                else:
                    self.buckets[index] = current.next
                self.count -= 1
                return True
            prev = current
            current = current.next
        
        return False
    
    def load_factor(self) -> float:
        """Return current load factor (elements / buckets)."""
        return self.count / self.size
    
    def display(self):
        """Print visual representation of the hash table."""
        print(f"Hash Table (size={self.size}, count={self.count}, load={self.load_factor():.2f})")
        print("=" * 50)
        for i in range(self.size):
            chain = []
            current = self.buckets[i]
            while current:
                chain.append(f"({current.key}: {current.value})")
                current = current.next
            chain_str = " → ".join(chain) if chain else "empty"
            print(f"[{i}]: {chain_str}")
    
    def __getitem__(self, key):
        return self.get(key)
    
    def __setitem__(self, key, value):
        self.put(key, value)


# Demonstrate chaining
print("=== Hash Table with Chaining ===")
ht = HashTableChaining(7)

# Insert values that will cause collisions
data = [(15, "apple"), (22, "banana"), (8, "cherry"), 
        (1, "date"), (29, "elderberry"), (36, "fig")]

print("\nInserting keys and their hash values (mod 7):")
for key, value in data:
    print(f"  {key} % 7 = {key % 7}")
    ht.put(key, value)

print("\nResulting hash table:")
ht.display()

print(f"\nLookup examples:")
print(f"  get(22) = {ht.get(22)}")
print(f"  get(36) = {ht.get(36)}")
print(f"  get(100) = {ht.get(100)}")

In [ ]:
class HashTableLinearProbing:
    """
    Hash table with linear probing collision resolution.
    
    On collision, linearly search for the next empty slot.
    Probe sequence: h(k), h(k)+1, h(k)+2, ...
    
    Attributes:
        size: Number of slots in the table
        keys: Array of keys
        values: Array of values
        count: Number of key-value pairs stored
        DELETED: Sentinel value for deleted slots (tombstone)
    """
    
    DELETED = object()  # Tombstone marker
    
    def __init__(self, size: int = 11):
        """
        Initialize hash table with given size.
        
        Args:
            size: Number of slots (should be prime for better distribution)
        """
        self.size = size
        self.keys = [None] * size
        self.values = [None] * size
        self.count = 0
    
    def _hash(self, key) -> int:
        """Compute primary hash index."""
        return hash(key) % self.size
    
    def put(self, key, value) -> bool:
        """
        Insert or update a key-value pair using linear probing.
        
        Args:
            key: Key to insert
            value: Value associated with the key
        
        Returns:
            True if successful, False if table is full
        """
        if self.count >= self.size:
            return False  # Table full
        
        index = self._hash(key)
        start = index
        first_deleted = None
        
        while True:
            if self.keys[index] is None:
                # Found empty slot
                insert_idx = first_deleted if first_deleted is not None else index
                self.keys[insert_idx] = key
                self.values[insert_idx] = value
                self.count += 1
                return True
            
            if self.keys[index] is self.DELETED:
                # Remember first deleted slot for insertion
                if first_deleted is None:
                    first_deleted = index
            elif self.keys[index] == key:
                # Key exists, update value
                self.values[index] = value
                return True
            
            index = (index + 1) % self.size
            if index == start:
                # Table full (shouldn't happen with count check)
                if first_deleted is not None:
                    self.keys[first_deleted] = key
                    self.values[first_deleted] = value
                    self.count += 1
                    return True
                return False
    
    def get(self, key):
        """
        Retrieve value by key using linear probing.
        
        Args:
            key: Key to search for
        
        Returns:
            Value if found, None otherwise
        """
        index = self._hash(key)
        start = index
        
        while self.keys[index] is not None:
            if self.keys[index] != self.DELETED and self.keys[index] == key:
                return self.values[index]
            index = (index + 1) % self.size
            if index == start:
                break
        
        return None
    
    def delete(self, key) -> bool:
        """
        Remove a key using tombstone marker.
        
        Args:
            key: Key to remove
        
        Returns:
            True if key was found and removed, False otherwise
        """
        index = self._hash(key)
        start = index
        
        while self.keys[index] is not None:
            if self.keys[index] != self.DELETED and self.keys[index] == key:
                self.keys[index] = self.DELETED
                self.values[index] = None
                self.count -= 1
                return True
            index = (index + 1) % self.size
            if index == start:
                break
        
        return False
    
    def load_factor(self) -> float:
        """Return current load factor."""
        return self.count / self.size
    
    def display(self):
        """Print visual representation of the hash table."""
        print(f"Hash Table - Linear Probing (size={self.size}, count={self.count}, load={self.load_factor():.2f})")
        print("=" * 60)
        
        # Header
        header = "Index: "
        for i in range(self.size):
            header += f"{i:^6}"
        print(header)
        
        # Border
        border = "      ┌" + "─────┬" * (self.size - 1) + "─────┐"
        print(border)
        
        # Values
        values = "      │"
        for i in range(self.size):
            if self.keys[i] is None:
                values += "     │"
            elif self.keys[i] is self.DELETED:
                values += "  X  │"
            else:
                values += f"{self.keys[i]:^5}│"
        print(values)
        
        # Bottom border
        bottom = "      └" + "─────┴" * (self.size - 1) + "─────┘"
        print(bottom)


# Demonstrate linear probing
print("=== Hash Table with Linear Probing ===")
print("\nInserting 15, 22, 29 (all hash to same index with mod 7):")

ht = HashTableLinearProbing(7)

# Show step by step
keys = [15, 22, 29]
for key in keys:
    print(f"\nInsert {key}: h({key}) = {key % 7}")
    ht.put(key, f"val_{key}")
    ht.display()

print(f"\nLookup: get(22) = {ht.get(22)}")
print(f"Lookup: get(29) = {ht.get(29)}")

In [ ]:
class HashTableQuadraticProbing:
    """
    Hash table with quadratic probing collision resolution.
    
    Probe sequence: h(k), h(k)+1², h(k)+2², h(k)+3², ...
    Reduces primary clustering compared to linear probing.
    
    Note: Table size should be prime and load factor < 0.5
    to guarantee finding an empty slot.
    """
    
    DELETED = object()
    
    def __init__(self, size: int = 11):
        """
        Initialize hash table.
        
        Args:
            size: Table size (should be prime)
        """
        self.size = size
        self.keys = [None] * size
        self.values = [None] * size
        self.count = 0
    
    def _hash(self, key) -> int:
        """Compute primary hash index."""
        return hash(key) % self.size
    
    def _probe(self, key, i: int) -> int:
        """
        Compute probe position for attempt i.
        
        Args:
            key: Key being inserted/searched
            i: Probe attempt number (0, 1, 2, ...)
        
        Returns:
            Index to probe
        """
        return (self._hash(key) + i * i) % self.size
    
    def put(self, key, value) -> bool:
        """
        Insert or update using quadratic probing.
        
        Returns:
            True if successful, False if no slot found
        """
        first_deleted = None
        
        for i in range(self.size):
            index = self._probe(key, i)
            
            if self.keys[index] is None:
                insert_idx = first_deleted if first_deleted is not None else index
                self.keys[insert_idx] = key
                self.values[insert_idx] = value
                self.count += 1
                return True
            
            if self.keys[index] is self.DELETED:
                if first_deleted is None:
                    first_deleted = index
            elif self.keys[index] == key:
                self.values[index] = value
                return True
        
        # If we found a deleted slot, use it
        if first_deleted is not None:
            self.keys[first_deleted] = key
            self.values[first_deleted] = value
            self.count += 1
            return True
        
        return False
    
    def get(self, key):
        """Retrieve value using quadratic probing."""
        for i in range(self.size):
            index = self._probe(key, i)
            
            if self.keys[index] is None:
                return None
            if self.keys[index] != self.DELETED and self.keys[index] == key:
                return self.values[index]
        
        return None
    
    def display(self):
        """Print visual representation."""
        print(f"Quadratic Probing (size={self.size}, count={self.count})")
        for i in range(self.size):
            if self.keys[i] is None:
                print(f"  [{i}]: empty")
            elif self.keys[i] is self.DELETED:
                print(f"  [{i}]: DELETED")
            else:
                print(f"  [{i}]: {self.keys[i]} → {self.values[i]}")


# Demonstrate quadratic probing
print("=== Quadratic Probing Demo ===")
print("\nProbe sequence for h(k)=1: 1, 1+1=2, 1+4=5, 1+9=10, 1+16=6 (mod 11)")

ht = HashTableQuadraticProbing(11)

# Insert values that collide
for key in [12, 23, 34, 45]:  # All hash to 1 (mod 11)
    ht.put(key, f"val_{key}")

print("\nAfter inserting 12, 23, 34, 45 (all hash to 1):")
ht.display()

In [ ]:
class HashTableDoubleHashing:
    """
    Hash table with double hashing collision resolution.
    
    Uses two hash functions:
        h1(k) = k mod m (primary hash)
        h2(k) = 1 + (k mod (m-1)) (step size, never 0)
    
    Probe sequence: h1(k), h1(k)+h2(k), h1(k)+2*h2(k), ...
    Provides best distribution among open addressing methods.
    """
    
    DELETED = object()
    
    def __init__(self, size: int = 11):
        """
        Initialize hash table.
        
        Args:
            size: Table size (should be prime)
        """
        self.size = size
        self.keys = [None] * size
        self.values = [None] * size
        self.count = 0
    
    def _hash1(self, key) -> int:
        """Primary hash function."""
        return hash(key) % self.size
    
    def _hash2(self, key) -> int:
        """
        Secondary hash function for step size.
        Returns value in range [1, size-1], never 0.
        """
        return 1 + (hash(key) % (self.size - 1))
    
    def _probe(self, key, i: int) -> int:
        """Compute probe position for attempt i."""
        return (self._hash1(key) + i * self._hash2(key)) % self.size
    
    def put(self, key, value) -> bool:
        """Insert or update using double hashing."""
        first_deleted = None
        
        for i in range(self.size):
            index = self._probe(key, i)
            
            if self.keys[index] is None:
                insert_idx = first_deleted if first_deleted is not None else index
                self.keys[insert_idx] = key
                self.values[insert_idx] = value
                self.count += 1
                return True
            
            if self.keys[index] is self.DELETED:
                if first_deleted is None:
                    first_deleted = index
            elif self.keys[index] == key:
                self.values[index] = value
                return True
        
        if first_deleted is not None:
            self.keys[first_deleted] = key
            self.values[first_deleted] = value
            self.count += 1
            return True
        
        return False
    
    def get(self, key):
        """Retrieve value using double hashing."""
        for i in range(self.size):
            index = self._probe(key, i)
            
            if self.keys[index] is None:
                return None
            if self.keys[index] != self.DELETED and self.keys[index] == key:
                return self.values[index]
        
        return None
    
    def display(self):
        """Print visual representation."""
        print(f"Double Hashing (size={self.size}, count={self.count})")
        for i in range(self.size):
            if self.keys[i] is None:
                print(f"  [{i}]: empty")
            elif self.keys[i] is self.DELETED:
                print(f"  [{i}]: DELETED")
            else:
                print(f"  [{i}]: {self.keys[i]} → {self.values[i]}")


# Demonstrate double hashing
print("=== Double Hashing Demo ===")

ht = HashTableDoubleHashing(11)

# Show how step size varies by key
print("\nStep sizes for different keys:")
for key in [12, 23, 34, 45]:
    h1 = hash(key) % 11
    h2 = 1 + (hash(key) % 10)
    print(f"  key={key}: h1={h1}, h2={h2} → probes: {h1}, {(h1+h2)%11}, {(h1+2*h2)%11}, ...")
    ht.put(key, f"val_{key}")

print("\nResulting table:")
ht.display()

---

## 4. Load Factor & Rehashing

### Load Factor

The **load factor** α = n/m where:
- n = number of elements stored
- m = number of slots (table size)

```
Load Factor Effects:

α = 0.25 (25% full)     α = 0.75 (75% full)     α = 0.95 (95% full)
┌─┬─┬─┬─┬─┬─┬─┬─┐       ┌─┬─┬─┬─┬─┬─┬─┬─┐       ┌─┬─┬─┬─┬─┬─┬─┬─┐
│■│ │ │ │■│ │ │ │       │■│■│■│ │■│■│ │■│       │■│■│■│■│■│■│■│ │
└─┴─┴─┴─┴─┴─┴─┴─┘       └─┴─┴─┴─┴─┴─┴─┴─┘       └─┴─┴─┴─┴─┴─┴─┴─┘
Fast operations         Good balance            Slow! Many collisions
```

### Performance vs Load Factor

| Load Factor | Chaining (avg probes) | Linear Probing (avg probes) |
|-------------|----------------------|-----------------------------|
| 0.5 | 1.25 | 1.5 |
| 0.75 | 1.375 | 2.5 |
| 0.9 | 1.45 | 5.5 |
| 0.95 | 1.475 | 10.5 |

### When to Rehash

**Typical thresholds:**
- Chaining: rehash when α > 0.75 or > 1.0
- Open addressing: rehash when α > 0.5 or > 0.7

**Rehashing process:**
1. Create new table (typically 2x size, round to next prime)
2. Recompute hash for each element
3. Insert into new table
4. Replace old table with new one

**Amortized cost:** O(1) per operation (occasional O(n) rehash is spread across many insertions)

In [ ]:
def is_prime(n: int) -> bool:
    """Check if n is prime."""
    if n < 2:
        return False
    if n == 2:
        return True
    if n % 2 == 0:
        return False
    for i in range(3, int(n**0.5) + 1, 2):
        if n % i == 0:
            return False
    return True


def next_prime(n: int) -> int:
    """Find the smallest prime >= n."""
    while not is_prime(n):
        n += 1
    return n


class HashTableWithRehashing:
    """
    Hash table with automatic rehashing when load factor exceeds threshold.
    
    Uses chaining for collision resolution.
    Automatically doubles size (to next prime) when load > threshold.
    
    Attributes:
        size: Current number of buckets
        buckets: Array of linked list heads
        count: Number of elements stored
        load_threshold: Maximum load factor before rehashing
    """
    
    def __init__(self, initial_size: int = 7, load_threshold: float = 0.75):
        """
        Initialize hash table with rehashing support.
        
        Args:
            initial_size: Starting number of buckets
            load_threshold: Trigger rehash when load exceeds this
        """
        self.size = next_prime(initial_size)
        self.buckets = [None] * self.size
        self.count = 0
        self.load_threshold = load_threshold
        self.rehash_count = 0
    
    def _hash(self, key) -> int:
        """Compute bucket index."""
        return hash(key) % self.size
    
    def load_factor(self) -> float:
        """Current load factor."""
        return self.count / self.size
    
    def _rehash(self):
        """
        Double the table size and rehash all elements.
        
        New size is the next prime after 2*current_size.
        """
        old_buckets = self.buckets
        old_size = self.size
        
        # Create new larger table
        self.size = next_prime(2 * old_size)
        self.buckets = [None] * self.size
        self.count = 0
        self.rehash_count += 1
        
        print(f"  → Rehashing: {old_size} → {self.size} buckets")
        
        # Reinsert all elements
        for bucket in old_buckets:
            current = bucket
            while current:
                self._insert_no_rehash(current.key, current.value)
                current = current.next
    
    def _insert_no_rehash(self, key, value):
        """Insert without triggering rehash (used during rehashing)."""
        index = self._hash(key)
        
        # Check for existing key
        current = self.buckets[index]
        while current:
            if current.key == key:
                current.value = value
                return
            current = current.next
        
        # Insert at head
        new_node = Node(key, value)
        new_node.next = self.buckets[index]
        self.buckets[index] = new_node
        self.count += 1
    
    def put(self, key, value):
        """
        Insert key-value pair, rehashing if needed.
        
        Args:
            key: Key to insert
            value: Value associated with key
        """
        # Check if rehash needed
        if self.load_factor() >= self.load_threshold:
            self._rehash()
        
        self._insert_no_rehash(key, value)
    
    def get(self, key):
        """Retrieve value by key."""
        index = self._hash(key)
        current = self.buckets[index]
        
        while current:
            if current.key == key:
                return current.value
            current = current.next
        
        return None
    
    def stats(self):
        """Print table statistics."""
        print(f"Size: {self.size}, Count: {self.count}")
        print(f"Load factor: {self.load_factor():.3f}")
        print(f"Rehash operations: {self.rehash_count}")


# Demonstrate rehashing
print("=== Automatic Rehashing Demo ===")
print("Initial size: 7, threshold: 0.75\n")

ht = HashTableWithRehashing(initial_size=7, load_threshold=0.75)

# Insert elements and watch for rehashing
for i in range(20):
    ht.put(f"key_{i}", f"value_{i}")
    if i % 5 == 4:  # Print stats every 5 insertions
        print(f"After {i+1} insertions: size={ht.size}, load={ht.load_factor():.3f}")

print("\nFinal statistics:")
ht.stats()
print(f"\nLookup test: get('key_15') = {ht.get('key_15')}")

---

## 5. Bloom Filters

### What is a Bloom Filter?

A **Bloom filter** is a space-efficient probabilistic data structure for set membership testing.

**Key properties:**
- **No false negatives**: If "not in set", it's definitely not there
- **Possible false positives**: If "in set", it's probably there (small chance of error)
- **No deletion**: Standard Bloom filters don't support removal
- **Very compact**: Uses only m bits regardless of element size

### How It Works

```
Bloom Filter with k=3 hash functions, m=10 bits

Initial state (all zeros):
Bit index:  0   1   2   3   4   5   6   7   8   9
           [0] [0] [0] [0] [0] [0] [0] [0] [0] [0]

Insert "hello":
  h1("hello") = 2  →  set bit 2
  h2("hello") = 5  →  set bit 5
  h3("hello") = 8  →  set bit 8

After insert:
           [0] [0] [1] [0] [0] [1] [0] [0] [1] [0]
                    ↑           ↑           ↑
                   h1          h2          h3

Insert "world":
  h1("world") = 1  →  set bit 1
  h2("world") = 4  →  set bit 4
  h3("world") = 8  →  bit 8 already set!

After insert:
           [0] [1] [1] [0] [1] [1] [0] [0] [1] [0]
                ↑   ↑       ↑   ↑           ↑
          world↑   ↑hello   ↑   ↑hello      both

Query "apple":
  h1("apple") = 3  →  bit[3]=0  →  DEFINITELY NOT IN SET ✓

Query "hello":
  h1("hello") = 2  →  bit[2]=1  ✓
  h2("hello") = 5  →  bit[5]=1  ✓
  h3("hello") = 8  →  bit[8]=1  ✓
  All bits set → PROBABLY IN SET ✓

Query "test" (false positive example):
  h1("test") = 2  →  bit[2]=1  ✓ (set by "hello")
  h2("test") = 5  →  bit[5]=1  ✓ (set by "hello")
  h3("test") = 8  →  bit[8]=1  ✓ (set by both)
  All bits set → PROBABLY IN SET (but it's not! FALSE POSITIVE)
```

### Complexity

| Operation | Time | Space |
|-----------|------|-------|
| Insert | O(k) | O(m) bits |
| Lookup | O(k) | - |
| Delete | N/A | - |

Where:
- k = number of hash functions
- m = number of bits in array
- n = number of elements to insert

### False Positive Probability

```
P(false positive) ≈ (1 - e^(-kn/m))^k

Optimal k = (m/n) * ln(2) ≈ 0.693 * (m/n)
```

| m/n ratio | Optimal k | False positive rate |
|-----------|-----------|--------------------|
| 8 | 6 | 2.2% |
| 10 | 7 | 0.8% |
| 16 | 11 | 0.05% |
| 20 | 14 | 0.006% |

### Use Cases

1. **Web browsers**: Check if URL is in malware blacklist
2. **Databases**: Avoid disk reads for non-existent keys
3. **Spell checkers**: Quick "definitely misspelled" check
4. **Network routers**: Packet deduplication

In [ ]:
import math
from typing import List, Callable


class BloomFilter:
    """
    Bloom filter for probabilistic set membership testing.
    
    A space-efficient data structure that can tell you:
    - "Definitely not in set" (no false negatives)
    - "Probably in set" (possible false positives)
    
    Attributes:
        size: Number of bits in the bit array (m)
        num_hashes: Number of hash functions (k)
        bits: The bit array
        count: Number of elements inserted
    """
    
    def __init__(self, size: int = 1000, num_hashes: int = 3):
        """
        Initialize Bloom filter.
        
        Args:
            size: Number of bits in the filter (m)
            num_hashes: Number of hash functions to use (k)
        """
        self.size = size
        self.num_hashes = num_hashes
        self.bits = [0] * size
        self.count = 0
    
    def _hashes(self, item) -> List[int]:
        """
        Generate k hash values for an item.
        
        Uses double hashing: h_i(x) = (h1(x) + i * h2(x)) % m
        This generates k different hash values from two base hashes.
        
        Args:
            item: Item to hash
        
        Returns:
            List of k hash indices
        """
        # Use Python's built-in hash and a modified version
        h1 = hash(item) % self.size
        h2 = (hash(str(item) + "salt") % (self.size - 1)) + 1
        
        return [(h1 + i * h2) % self.size for i in range(self.num_hashes)]
    
    def add(self, item) -> None:
        """
        Add an item to the Bloom filter.
        
        Sets the bits at all k hash positions to 1.
        
        Args:
            item: Item to add
        """
        for index in self._hashes(item):
            self.bits[index] = 1
        self.count += 1
    
    def contains(self, item) -> bool:
        """
        Check if an item might be in the set.
        
        Args:
            item: Item to check
        
        Returns:
            False = definitely not in set
            True = probably in set (may be false positive)
        """
        return all(self.bits[index] == 1 for index in self._hashes(item))
    
    def false_positive_probability(self) -> float:
        """
        Estimate current false positive probability.
        
        Formula: (1 - e^(-kn/m))^k
        
        Returns:
            Estimated false positive rate
        """
        if self.count == 0:
            return 0.0
        
        k = self.num_hashes
        m = self.size
        n = self.count
        
        # P = (1 - e^(-kn/m))^k
        return (1 - math.exp(-k * n / m)) ** k
    
    def fill_ratio(self) -> float:
        """Return the fraction of bits set to 1."""
        return sum(self.bits) / self.size
    
    def display(self, width: int = 50):
        """
        Print a visual representation of the bit array.
        
        Args:
            width: Maximum characters per line
        """
        print(f"Bloom Filter: {self.size} bits, {self.num_hashes} hash functions")
        print(f"Elements: {self.count}, Fill ratio: {self.fill_ratio():.1%}")
        print(f"Estimated FP rate: {self.false_positive_probability():.4%}")
        
        # Visual representation (truncated for large filters)
        display_size = min(self.size, width)
        bit_str = ''.join('█' if b else '░' for b in self.bits[:display_size])
        if self.size > width:
            bit_str += f"... ({self.size - width} more bits)"
        print(f"Bits: {bit_str}")
    
    def __contains__(self, item) -> bool:
        """Support 'in' operator."""
        return self.contains(item)


# Demonstrate Bloom Filter
print("=== Bloom Filter Demo ===")

# Create a small Bloom filter for demonstration
bf = BloomFilter(size=20, num_hashes=3)

# Add some elements
words = ["hello", "world", "bloom", "filter"]
print("\nInserting:", words)

for word in words:
    print(f"\n  Adding '{word}':")
    hashes = bf._hashes(word)
    print(f"    Hash positions: {hashes}")
    bf.add(word)
    bf.display()

print("\n" + "="*50)
print("Membership queries:")

# Test membership
test_words = ["hello", "world", "python", "java", "bloom"]
for word in test_words:
    result = "MAYBE" if word in bf else "NO"
    actual = "(inserted)" if word in words else "(not inserted)"
    status = "✓" if (result == "MAYBE") == (word in words) else "FALSE POSITIVE!"
    print(f"  '{word}' in filter? {result} {actual} {status}")

In [ ]:
import random
import string


def demonstrate_false_positives():
    """
    Demonstrate Bloom filter false positives empirically.
    
    Inserts n random strings, then tests m random non-inserted strings
    to measure actual false positive rate.
    """
    print("=== False Positive Rate Analysis ===")
    
    # Test different configurations
    configs = [
        {"size": 1000, "num_hashes": 3, "n_elements": 100},
        {"size": 1000, "num_hashes": 7, "n_elements": 100},
        {"size": 10000, "num_hashes": 7, "n_elements": 1000},
    ]
    
    def random_string(length=10):
        return ''.join(random.choices(string.ascii_lowercase, k=length))
    
    print(f"\n{'Config':<30} {'Theory FP%':>12} {'Actual FP%':>12} {'Fill%':>10}")
    print("-" * 70)
    
    for config in configs:
        bf = BloomFilter(size=config["size"], num_hashes=config["num_hashes"])
        
        # Insert n random strings
        inserted = set()
        for _ in range(config["n_elements"]):
            s = random_string()
            inserted.add(s)
            bf.add(s)
        
        # Test with strings we know are NOT in the set
        n_tests = 10000
        false_positives = 0
        
        for _ in range(n_tests):
            test_str = random_string(15)  # Different length to avoid collision
            if test_str not in inserted and bf.contains(test_str):
                false_positives += 1
        
        actual_fp_rate = false_positives / n_tests
        theoretical_fp = bf.false_positive_probability()
        
        config_str = f"m={config['size']}, k={config['num_hashes']}, n={config['n_elements']}"
        print(f"{config_str:<30} {theoretical_fp*100:>11.2f}% {actual_fp_rate*100:>11.2f}% {bf.fill_ratio()*100:>9.1f}%")
    
    print("\nObservations:")
    print("- More hash functions (k) generally reduces FP rate up to a point")
    print("- Larger filter (m) reduces FP rate")
    print("- Actual rates approximate theoretical values")


demonstrate_false_positives()

In [ ]:
def optimal_bloom_filter_params(n: int, target_fp: float) -> dict:
    """
    Calculate optimal Bloom filter parameters.
    
    Given the expected number of elements and desired false positive rate,
    computes the optimal number of bits (m) and hash functions (k).
    
    Args:
        n: Expected number of elements to insert
        target_fp: Desired false positive rate (e.g., 0.01 for 1%)
    
    Returns:
        Dictionary with optimal m, k, and bits per element
    """
    # Optimal m = -(n * ln(p)) / (ln(2)^2)
    m = int(-n * math.log(target_fp) / (math.log(2) ** 2))
    
    # Optimal k = (m/n) * ln(2)
    k = max(1, int((m / n) * math.log(2)))
    
    # Actual FP rate with these parameters
    actual_fp = (1 - math.exp(-k * n / m)) ** k
    
    return {
        "num_bits": m,
        "num_hashes": k,
        "bits_per_element": m / n,
        "actual_fp_rate": actual_fp,
        "memory_bytes": m // 8
    }


print("=== Bloom Filter Sizing Guide ===")
print("\nRecommended parameters for common scenarios:\n")

scenarios = [
    (1000, 0.01, "Small set, 1% FP"),
    (10000, 0.01, "Medium set, 1% FP"),
    (1000000, 0.01, "Large set, 1% FP"),
    (1000000, 0.001, "Large set, 0.1% FP"),
    (1000000, 0.0001, "Large set, 0.01% FP"),
]

print(f"{'Scenario':<25} {'n':>10} {'m (bits)':>12} {'k':>5} {'Bits/elem':>10} {'Memory':>10}")
print("-" * 80)

for n, fp, desc in scenarios:
    params = optimal_bloom_filter_params(n, fp)
    
    # Format memory
    mem = params['memory_bytes']
    if mem < 1024:
        mem_str = f"{mem} B"
    elif mem < 1024 * 1024:
        mem_str = f"{mem/1024:.1f} KB"
    else:
        mem_str = f"{mem/(1024*1024):.1f} MB"
    
    print(f"{desc:<25} {n:>10,} {params['num_bits']:>12,} {params['num_hashes']:>5} {params['bits_per_element']:>10.1f} {mem_str:>10}")

---

## Summary

### Hash Table Complexity

| Operation | Average | Worst | Notes |
|-----------|---------|-------|-------|
| Insert | **O(1)** | O(n) | Amortized with rehashing |
| Search | **O(1)** | O(n) | Worst case: all keys collide |
| Delete | **O(1)** | O(n) | Open addressing needs tombstones |
| Space | O(n) | O(n) | Plus overhead for load factor |

### Collision Resolution Comparison

| Method | Pros | Cons |
|--------|------|------|
| **Chaining** | Simple, load > 1 OK, easy delete | Extra memory, cache unfriendly |
| **Linear Probing** | Cache friendly, simple | Clustering, delete complex |
| **Quadratic Probing** | Less clustering | May not find slot, delete complex |
| **Double Hashing** | Best distribution | Two hash functions, slower |

### Bloom Filter Complexity

| Operation | Time | Space |
|-----------|------|-------|
| Insert | O(k) | m bits total |
| Lookup | O(k) | - |
| Space/element | - | ~10 bits for 1% FP |

### When to Use What

- **Hash Table**: Need exact lookups, insertions, deletions
- **Bloom Filter**: Need fast "definitely not" answers, memory constrained, can tolerate false positives

### Key Takeaways

1. **Hash functions** should be fast, deterministic, and distribute keys uniformly
2. **Load factor** critically affects performance - keep it below 0.75 (chaining) or 0.5 (open addressing)
3. **Rehashing** maintains O(1) amortized operations as table grows
4. **Bloom filters** trade certainty for massive space savings

In [ ]:
# Comparison of all hash table implementations
import time


def benchmark_hash_tables(n_operations: int = 10000):
    """
    Benchmark different hash table implementations.
    
    Args:
        n_operations: Number of insert and lookup operations to perform
    """
    print(f"=== Hash Table Benchmark ({n_operations:,} operations) ===")
    
    # Generate test data
    test_keys = [f"key_{i}" for i in range(n_operations)]
    test_values = [f"value_{i}" for i in range(n_operations)]
    
    implementations = [
        ("Python dict", dict),
        ("Chaining", lambda: HashTableChaining(1009)),  # Prime size
        ("Linear Probing", lambda: HashTableLinearProbing(n_operations * 2)),
        ("With Rehashing", lambda: HashTableWithRehashing(101, 0.75)),
    ]
    
    print(f"\n{'Implementation':<20} {'Insert (ms)':>12} {'Lookup (ms)':>12} {'Total (ms)':>12}")
    print("-" * 60)
    
    for name, factory in implementations:
        ht = factory()
        
        # Benchmark inserts
        start = time.perf_counter()
        for k, v in zip(test_keys, test_values):
            if isinstance(ht, dict):
                ht[k] = v
            else:
                ht.put(k, v)
        insert_time = (time.perf_counter() - start) * 1000
        
        # Benchmark lookups
        start = time.perf_counter()
        for k in test_keys:
            if isinstance(ht, dict):
                _ = ht.get(k)
            else:
                _ = ht.get(k)
        lookup_time = (time.perf_counter() - start) * 1000
        
        total = insert_time + lookup_time
        print(f"{name:<20} {insert_time:>12.2f} {lookup_time:>12.2f} {total:>12.2f}")
    
    print("\nNote: Python's built-in dict is highly optimized C code.")
    print("Our implementations are for educational purposes.")


benchmark_hash_tables(5000)

---

[← Previous: Red-Black Trees](../../Tier_4_Algorithms_and_Data_Structures/05_Tree_Structures/03_red_black_trees.ipynb) | [Next: Naive Pattern Matching Algorithm →](../../Tier_4_Algorithms_and_Data_Structures/07_String_Algorithms/01_naive_pattern_matching.ipynb)